# Origin — Evaluation Dataset Preparation

This notebook prepares and validates the evaluation query dataset that will later be used to compare the TF-IDF, semantic, and hybrid retrieval approaches.

The evaluation dataset contains realistic Arabic and English graduation-project proposals and is kept separate from the historical project database.

## 1. Setup and Load Evaluation Dataset

In [1]:
import pandas as pd

EVAL_DATA_URL = (
    "https://raw.githubusercontent.com/"
    "esrrahafase-crypto/Origin/main/"
    "data/evaluation/evaluation_queries.csv"
)

eval_df = pd.read_csv(EVAL_DATA_URL, encoding="utf-8-sig")

print("ORIGIN | EVALUATION DATASET")
print("=" * 55)
print("Dataset loaded successfully.")
print("Shape:", eval_df.shape)
print("Columns:", list(eval_df.columns))

display(eval_df.head())

ORIGIN | EVALUATION DATASET
Dataset loaded successfully.
Shape: (60, 7)
Columns: ['query_id', 'query_title', 'query_description', 'language', 'generation_intent', 'reference_project_ids', 'split']


,query_id,query_title,query_description,language,generation_intent,reference_project_ids,split
0,Q001,Balcony Irrigation Companion,Apartment residents often lose balcony plants ...,en,near_overlap,P001,final_test
1,Q002,Shared Coding Studio for Capstone Teams,Student teams waste time reconciling code edit...,en,near_overlap,P029;P008,final_test
2,Q003,Blood Stock Planning Desk,Hospital blood-bank staff need earlier notice ...,en,near_overlap,P059,dev
3,Q004,Everyday Sound Notifier,Deaf residents may miss a doorbell or househol...,en,near_overlap,P074,dev
4,Q005,Suspicious Link Checkpoint,People following links from shopping messages ...,en,near_overlap,P101,final_test


## 2. Validate Dataset Structure and Distribution


In [2]:
# Basic dataset validation

expected_columns = [
    "query_id",
    "query_title",
    "query_description",
    "language",
    "generation_intent",
    "reference_project_ids",
    "split"
]

print("ORIGIN | EVALUATION DATASET VALIDATION")
print("=" * 60)

# Shape
print("Total queries:", len(eval_df))
print("Total columns:", len(eval_df.columns))

# Schema
missing_columns = [c for c in expected_columns if c not in eval_df.columns]
extra_columns = [c for c in eval_df.columns if c not in expected_columns]

print("\nMissing columns:", missing_columns)
print("Unexpected columns:", extra_columns)

# Missing values
print("\nMissing values:")
print(eval_df.isna().sum())

# Duplicate checks
print("\nDuplicate query IDs:", eval_df["query_id"].duplicated().sum())
print("Duplicate titles:", eval_df["query_title"].duplicated().sum())
print("Duplicate descriptions:", eval_df["query_description"].duplicated().sum())

# Distributions
print("\nLanguage distribution:")
print(eval_df["language"].value_counts())

print("\nGeneration intent distribution:")
print(eval_df["generation_intent"].value_counts())

print("\nSplit distribution:")
print(eval_df["split"].value_counts())

ORIGIN | EVALUATION DATASET VALIDATION
Total queries: 60
Total columns: 7

Missing columns: []
Unexpected columns: []

Missing values:
query_id                  0
query_title               0
query_description         0
language                  0
generation_intent         0
reference_project_ids    20
split                     0
dtype: int64

Duplicate query IDs: 0
Duplicate titles: 0
Duplicate descriptions: 0

Language distribution:
language
en    30
ar    30
Name: count, dtype: int64

Generation intent distribution:
generation_intent
near_overlap        20
related_distinct    20
distant_novel       20
Name: count, dtype: int64

Split distribution:
split
dev           40
final_test    20
Name: count, dtype: int64


## 3. Validate Query IDs and Reference Project IDs

In [3]:
# Load historical project dataset

PROJECTS_URL = (
    "https://raw.githubusercontent.com/"
    "esrrahafase-crypto/Origin/main/"
    "data/processed/projects_clean.csv"
)

projects_df = pd.read_csv(PROJECTS_URL, encoding="utf-8-sig")

print("Historical projects shape:", projects_df.shape)

# Standardize column names in case the CSV uses capitalization
projects_df.columns = [c.strip().lower() for c in projects_df.columns]

print("Historical columns:", list(projects_df.columns))


Historical projects shape: (103, 7)
Historical columns: ['project_id', 'title', 'abstract', 'department', 'year', 'keywords', 'source_page']


In [4]:
# Validate query IDs

expected_query_ids = [f"Q{i:03d}" for i in range(1, 61)]
actual_query_ids = eval_df["query_id"].tolist()

missing_query_ids = sorted(set(expected_query_ids) - set(actual_query_ids))
unexpected_query_ids = sorted(set(actual_query_ids) - set(expected_query_ids))

print("Missing query IDs:", missing_query_ids)
print("Unexpected query IDs:", unexpected_query_ids)

# Validate reference project IDs

valid_project_ids = set(projects_df["project_id"].astype(str).str.strip())

invalid_reference_ids = []

for _, row in eval_df.iterrows():
    refs = row["reference_project_ids"]

    if pd.isna(refs):
        continue

    for ref in str(refs).split(";"):
        ref = ref.strip()

        if ref not in valid_project_ids:
            invalid_reference_ids.append({
                "query_id": row["query_id"],
                "invalid_project_id": ref
            })

print("\nInvalid reference project IDs:", len(invalid_reference_ids))

if invalid_reference_ids:
    display(pd.DataFrame(invalid_reference_ids))
else:
    print("All reference project IDs are valid.")

Missing query IDs: []
Unexpected query IDs: []

Invalid reference project IDs: 0
All reference project IDs are valid.


## 4. Validate Description Length, Language Consistency, and Leakage

In [5]:
import re

print("ORIGIN | QUERY QUALITY CHECKS")
print("=" * 60)

# --------------------------------------------------
# 1. Description length
# --------------------------------------------------

eval_df["description_word_count"] = (
    eval_df["query_description"]
    .astype(str)
    .str.split()
    .str.len()
)

print("Description length:")
print("Minimum:", eval_df["description_word_count"].min())
print("Maximum:", eval_df["description_word_count"].max())
print("Average:", round(eval_df["description_word_count"].mean(), 2))

print("\nLength groups:")
print(
    pd.cut(
        eval_df["description_word_count"],
        bins=[39, 60, 80, 100],
        labels=["40-60", "61-80", "81-100"],
        include_lowest=True
    ).value_counts().sort_index()
)

# --------------------------------------------------
# 2. Language consistency
# --------------------------------------------------

arabic_pattern = re.compile(r"[\u0600-\u06FF]")

def detect_script(text):
    text = str(text)
    arabic_chars = len(arabic_pattern.findall(text))
    latin_chars = len(re.findall(r"[A-Za-z]", text))

    if arabic_chars > latin_chars:
        return "ar"
    elif latin_chars > arabic_chars:
        return "en"
    else:
        return "unknown"

eval_df["detected_language"] = eval_df["query_description"].apply(detect_script)

language_mismatches = eval_df[
    eval_df["language"] != eval_df["detected_language"]
][
    ["query_id", "language", "detected_language", "query_title"]
]

print("\nLanguage mismatches:", len(language_mismatches))

if len(language_mismatches) > 0:
    display(language_mismatches)
else:
    print("All query language labels are consistent with the text.")

# --------------------------------------------------
# 3. Exact historical title leakage
# --------------------------------------------------

historical_titles = set(
    projects_df["title"]
    .astype(str)
    .str.strip()
    .str.lower()
)

eval_titles = (
    eval_df["query_title"]
    .astype(str)
    .str.strip()
    .str.lower()
)

exact_title_matches = eval_df[
    eval_titles.isin(historical_titles)
][["query_id", "query_title"]]

print("\nExact historical title matches:", len(exact_title_matches))

if len(exact_title_matches) > 0:
    display(exact_title_matches)
else:
    print("No evaluation query title exactly matches a historical project title.")

ORIGIN | QUERY QUALITY CHECKS
Description length:
Minimum: 50
Maximum: 99
Average: 74.8

Length groups:
description_word_count
40-60     20
61-80     20
81-100    20
Name: count, dtype: int64

Language mismatches: 0
All query language labels are consistent with the text.

Exact historical title matches: 0
No evaluation query title exactly matches a historical project title.


## 5. Final Evaluation Dataset Summary

In [6]:
print("ORIGIN | FINAL EVALUATION DATASET SUMMARY")
print("=" * 65)

summary = {
    "Total queries": len(eval_df),
    "English queries": (eval_df["language"] == "en").sum(),
    "Arabic queries": (eval_df["language"] == "ar").sum(),
    "Near-overlap queries": (eval_df["generation_intent"] == "near_overlap").sum(),
    "Related-distinct queries": (eval_df["generation_intent"] == "related_distinct").sum(),
    "Distant-novel queries": (eval_df["generation_intent"] == "distant_novel").sum(),
    "Development queries": (eval_df["split"] == "dev").sum(),
    "Final-test queries": (eval_df["split"] == "final_test").sum(),
    "Duplicate query IDs": eval_df["query_id"].duplicated().sum(),
    "Duplicate titles": eval_df["query_title"].duplicated().sum(),
    "Duplicate descriptions": eval_df["query_description"].duplicated().sum(),
    "Language mismatches": len(language_mismatches),
    "Exact historical title matches": len(exact_title_matches),
    "Invalid reference project IDs": len(invalid_reference_ids),
    "Minimum description words": eval_df["description_word_count"].min(),
    "Maximum description words": eval_df["description_word_count"].max(),
    "Average description words": round(
        eval_df["description_word_count"].mean(), 2
    )
}

summary_df = pd.DataFrame(
    summary.items(),
    columns=["Check", "Result"]
)

display(summary_df)

# Final readiness decision
critical_issues = (
    eval_df["query_id"].duplicated().sum()
    + eval_df["query_title"].duplicated().sum()
    + eval_df["query_description"].duplicated().sum()
    + len(language_mismatches)
    + len(exact_title_matches)
    + len(invalid_reference_ids)
)

print("\n" + "=" * 65)

if critical_issues == 0:
    print("✓ FINAL STATUS: READY FOR RETRIEVAL EXPERIMENTS")
    print(
        "The evaluation query dataset passed all critical validation checks."
    )
else:
    print("⚠ FINAL STATUS: REVIEW REQUIRED")
    print(
        f"{critical_issues} critical validation issue(s) were detected."
    )

ORIGIN | FINAL EVALUATION DATASET SUMMARY


,Check,Result
0,Total queries,60.0
1,English queries,30.0
2,Arabic queries,30.0
3,Near-overlap queries,20.0
4,Related-distinct queries,20.0
5,Distant-novel queries,20.0
6,Development queries,40.0
7,Final-test queries,20.0
8,Duplicate query IDs,0.0
9,Duplicate titles,0.0



✓ FINAL STATUS: READY FOR RETRIEVAL EXPERIMENTS
The evaluation query dataset passed all critical validation checks.
